In [2]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(100)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [3]:
#Define a dataclass for our message. This gives us a clear schema for each taxi trip:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

In [4]:
#Write a function to convert a DataFrame row into a Ride. 
#We convert the pandas Timestamp to epoch milliseconds - that's the format Flink expects later:
def ride_from_row(row):
    return Ride(
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        trip_distance=float(row['trip_distance']),
        total_amount=float(row['total_amount']),
        tpep_pickup_datetime=int(row['tpep_pickup_datetime'].timestamp() * 1000),
    )

In [5]:
ride = ride_from_row(df.iloc[0])
ride

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

Next, connect to Kafka. The bootstrap_servers is where the broker accepts connections - localhost:9092 because we're running this from our laptop (outside Docker). In production with multiple brokers, you'd list several for redundancy - if one is down, the client connects through another.

Kafka works with raw bytes, so we need a serializer that converts Python dicts to JSON:

In [6]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

Let's send a single ride to try it out. dataclasses.asdict(ride) converts the dataclass to a plain dict, which the serializer turns into JSON bytes. The broker auto-creates the rides topic on first use:

In [7]:
import dataclasses

topic_name = 'rides'

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

This works, but calling dataclasses.asdict() every time is tedious. We can make a serializer that handles dataclasses directly:

In [8]:
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

Now recreate the producer with the new serializer - we can pass Ride objects directly without converting them to dicts first:

In [9]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

#Send one ride to verify:
producer.send(topic_name, value=ride)
producer.flush()

In [10]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)
Sent: Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000)
Sent: Ride(PULocationID=163, DOLocationID=238, trip_distance=2.7, total_amount=25.62, tpep_pickup_datetime=1761955639000)
Sent: Ride(PULocationID=138, DOLocationID=261, trip_distance=12.87, total_amount=86.14, tpep_pickup_datetime=1761955200000)
Sent: Ride(PULocationID=138, DOLocationID=37, trip_distance=8.4, total_amount=48.65, tpep_pickup_datetime=1761956330000)
Sent: Ride(PULocationID=90, DOLocationID=100, trip_distance=0.85, total_amount=16.45, tpep_pickup_datetime=1761956471000)
Sent: Ride(PULocationID=142, DOLocationID=170, trip_distance=3.01, total_amount=25.85, tpep_pickup_datetime=1761955651000)
Sent: Ride(PULocationID=237, DOLocationID=144, trip_distance=3.82, total_amount=57.54, tpep_pickup_datetime=1761958012000)
Sent: Ride(PULocatio

If you're building from scratch (not using the cloned repo files), create the source directory structure and save the shared data model. The producer and consumer scripts both import from this file:

In [11]:
mkdir -p src/producers src/consumers src/job

Now let's read back the messages. The consumer receives raw bytes from Kafka. Instead of deserializing to a dict and then constructing a Ride manually, let's write a function that does both in one step:

In [11]:
import json

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

Test it with a sample JSON binary string (this is what Kafka delivers):

In [12]:
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'tpep_pickup_datetime': 1730429702000
}).encode('utf-8')

ride_deserializer(test_bytes)
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, tpep_pickup_datetime=1730429702000)

Now we can pass ride_deserializer directly as the value_deserializer - Kafka calls it on every message, so message.value is already a Ride.

Connect to Kafka as a consumer. auto_offset_reset='earliest' means we start reading from the beginning of the topic (without this, new consumers default to latest and only see new messages). group_id identifies this consumer group - Kafka tracks how far each group has read, so restarting with the same group ID continues where it left off:

In [13]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

Read messages and print them. Since value_deserializer returns a Ride, message.value is already a Ride object - no extra conversion needed:

In [14]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 18:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 18:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 18:13:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-10-31 18:49:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-10-31 18:07:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-10-31 18:00:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-10-31 18:18:50
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-10-31 18:21:11
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-10-31 18:07:31
Received: PU=237, DO=144, distance=3.82, amount=$57.54, pickup=2025-10-31 18:46:52

... received 10 messages so far (stopping after 10 for demo)
